# Notebook 08 — Leakage Quantification & Equal-Budget Topology Ablation

**Milestone 3 corrections (supervisor review, June 2026)**

## What this notebook does

| Section | What | Why |
|---------|------|-----|
| 1 | Apples-to-apples leakage inflation | Replace old 12-ep/batch-64 numbers with same CNN1D as NB04 |
| 2 | Equal-budget topology ablation | Single / dual / adaptive all at T=32, 80 epochs, patience=10 |
| 3 | Results compilation | Update `results/metrics_all.csv` with corrected rows |

### Novelty framing (after literature search)
Targeted searches on Google Scholar and arXiv for INCLUDE split leakage / duplicates found **no prior report** in papers that benchmark on INCLUDE (Sridhar 2020, Song 2025 HA-GCN, HWGAT).  
Claim carefully: *"To the best of our knowledge, no paper that benchmarks on INCLUDE has reported or corrected for cross-split video-ID duplicates in the HuggingFace parquet distribution."*

### Design rules
- **Inflation experiment**: same `CNN1D` as NB04, same `Adam lr=1e-3, batch=32, epochs=40, patience=8`. Only the split changes (clean SD vs raw HF parquet).
- **Topology ablation**: all three conditions at `T=32, SGD+Nesterov lr=0.01, cosine LR, batch=32, max 80 ep, patience=10`. Adaptive result from `retrain_best.py` is reused — no need to re-run it.
- `SEED = 42` locked throughout.

In [26]:
import sys, os, random, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader

ROOT     = Path("/Users/yamini/Desktop/projects/ISL PROJECT")
PROC_DIR = ROOT / "data" / "processed"
KP_DIR   = ROOT / "data" / "raw_keypoints"
PAR_DIR  = ROOT / "include_dataset" / "data"
RES_DIR  = ROOT / "results"
sys.path.insert(0, str(ROOT / "src"))

from graph   import build_adjacency
from model   import STGCN
from dataset import load_sd_loaders

SEED = 42

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    os.environ["PYTHONHASHSEED"] = str(s)

set_seed()

if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Seed   : {SEED}")

Device : mps
PyTorch: 2.8.0
Seed   : 42


---
## Section 1 — Leakage Quantification (apples-to-apples)

The HuggingFace INCLUDE parquet has duplicate video entries across splits.  
**28.4% of unique test videos** also appear in the raw train split.

**Empirical design:**  
Same `CNN1D` as NB04 (Adam, lr=1e-3, batch=32, epochs=40, patience=8, clip=1.0).  
Only the split varies:
- *Clean*: our deduplicated SD split — train=2,462 / val=332 / test=858, zero overlap  
- *Leaked*: raw HF parquet — train=3,292 / val=365 / test=887, ~30% of test seen in train  

`CLEAN_BASELINE = 0.865` is the locked NB04 SEED=42 result (no need to re-run it).

In [27]:
# ── 1a: Analytical overlap ────────────────────────────────────────────────────
df_tr = pq.read_table(PAR_DIR / "train-00000-of-00001.parquet").to_pandas()
df_va = pq.read_table(PAR_DIR / "val-00000-of-00001.parquet").to_pandas()
df_te = pq.read_table(PAR_DIR / "test-00000-of-00001.parquet").to_pandas()

print(f"Raw parquet sizes:  train={len(df_tr):,}  val={len(df_va):,}  test={len(df_te):,}")

train_ids = set(df_tr["video_path"].astype(str))
val_ids   = set(df_va["video_path"].astype(str))
test_ids  = set(df_te["video_path"].astype(str))

overlap  = train_ids & test_ids
leak_pct = 100 * len(overlap) / max(len(test_ids), 1)

print(f"\nCross-split overlaps (video_path level):")
print(f"  train ∩ val  : {len(train_ids & val_ids):4d}  ({100*len(train_ids & val_ids)/max(len(val_ids),1):.1f}% of unique val)")
print(f"  train ∩ test : {len(overlap):4d}  ({leak_pct:.1f}% of unique test)")
print(f"  val   ∩ test : {len(val_ids & test_ids):4d}  ({100*len(val_ids & test_ids)/max(len(test_ids),1):.1f}% of unique test)")
print(f"\n→ {leak_pct:.1f}% of test videos were also in train — fixed by NB02 priority dedup.")

Raw parquet sizes:  train=3,816  val=425  test=1,009

Cross-split overlaps (video_path level):
  train ∩ val  :  120  (29.1% of unique val)
  train ∩ test :  277  (28.4% of unique test)
  val   ∩ test :   27  (2.8% of unique test)

→ 28.4% of test videos were also in train — fixed by NB02 priority dedup.


In [28]:
# ── 1b: Preprocessing helpers (identical to NB02) ────────────────────────────
with open(PROC_DIR / "label_encoder.pkl", "rb") as f:
    le = pickle.load(f)
valid_labels = set(le.classes_)
L_SH, R_SH = 45, 46
T_FIXED = 64
FEAT_DIM = 53 * 3   # 159 flattened features per frame

def fix_gaps(kps):
    filled = kps.copy().astype(np.float32)
    T = kps.shape[0]
    for j in range(53):
        col = kps[:, j, :]
        det = ~np.all(col == 0, axis=1)
        if not det.any():
            continue
        vf = np.where(det)[0]
        for c in range(3):
            filled[:, j, c] = np.interp(np.arange(T, dtype=np.float64),
                                         vf.astype(np.float64), col[vf, c])
    return filled

def resample(kps, t_tgt=64):
    T = kps.shape[0]
    if T == t_tgt:
        return kps.astype(np.float32)
    src = np.arange(T, dtype=np.float32)
    tgt = np.linspace(0, T - 1, t_tgt, dtype=np.float32)
    out = np.zeros((t_tgt, 53, 3), dtype=np.float32)
    for j in range(53):
        for c in range(3):
            out[:, j, c] = np.interp(tgt, src, kps[:, j, c])
    return out

def normalize(kps):
    result = kps.copy()
    T = kps.shape[0]
    ls_all, rs_all = kps[:, L_SH, :], kps[:, R_SH, :]
    ok = ~np.all(ls_all == 0, axis=1)
    if ok.any():
        lsm, rsm = ls_all[ok].mean(0), rs_all[ok].mean(0)
        wm = np.linalg.norm(lsm[:2] - rsm[:2]) + 1e-6
    else:
        lsm = rsm = np.zeros(3); wm = 1.0
    for t in range(T):
        ls, rs = kps[t, L_SH, :], kps[t, R_SH, :]
        if np.all(ls == 0) and np.all(rs == 0):
            center, width = (lsm + rsm) / 2, wm
        else:
            center = (ls + rs) / 2
            width  = np.linalg.norm(ls[:2] - rs[:2]) + 1e-6
        result[t] = (kps[t] - center) / width
    return result

def vp_to_npy(vp):
    return vp.replace(".MOV","").replace(".mp4","").replace(".MP4","").replace("/","__") + ".npy"

def build_split(df_pq, name):
    rows = df_pq[df_pq["label"].isin(valid_labels)].reset_index(drop=True)
    Xl, yl, skip = [], [], 0
    t0 = time.time()
    for i, row in rows.iterrows():
        p = KP_DIR / vp_to_npy(row["video_path"])
        if not p.exists(): skip += 1; continue
        kps = np.load(str(p))
        if kps.shape[0] < 2: skip += 1; continue
        kps = fix_gaps(kps)
        kps = resample(kps, T_FIXED)
        kps = normalize(kps)
        Xl.append(kps)
        yl.append(le.transform([row["label"]])[0])
        if (i + 1) % 500 == 0:
            print(f"  [{name}] {i+1}/{len(rows)}  {time.time()-t0:.0f}s", flush=True)
    X = np.stack(Xl).astype(np.float32).reshape(len(Xl), T_FIXED, FEAT_DIM)
    y = np.array(yl, dtype=np.int64)
    print(f"  [{name}] done: {len(X):,} samples, {skip} skipped  ({time.time()-t0:.0f}s)")
    return X, y

print("Helpers defined.")

Helpers defined.


In [29]:
# ── 1c: Build leaked splits from raw parquet ──────────────────────────────────
# Preprocessing identical to NB02 (gap-fill → resample T=64 → torso-normalize).
# No deduplication — this is what a naive HuggingFace user gets.
print("Building leaked splits (gap-fill → resample → normalize) ...")
X_lk_tr, y_lk_tr = build_split(df_tr, "leaked-train")
X_lk_va, y_lk_va = build_split(df_va, "leaked-val")
X_lk_te, y_lk_te = build_split(df_te, "leaked-test")

# Load clean SD test split (already processed in NB02)
X_cl_te = np.load(PROC_DIR / "X_sd_test.npy").reshape(-1, T_FIXED, FEAT_DIM)
y_cl_te = np.load(PROC_DIR / "y_sd_test.npy")

print(f"\nLeaked : train={len(y_lk_tr):,}  val={len(y_lk_va):,}  test={len(y_lk_te):,}")
print(f"Clean  : test={len(y_cl_te):,}  (deduplicated SD split from NB02)")
print(f"\nNote: leaked test contains {len(overlap)} videos also in leaked train → inflation expected.")

Building leaked splits (gap-fill → resample → normalize) ...
  [leaked-train] 500/3810  1s
  [leaked-train] 1000/3810  2s
  [leaked-train] 2000/3810  4s
  [leaked-train] 2500/3810  5s
  [leaked-train] 3000/3810  6s
  [leaked-train] 3500/3810  7s
  [leaked-train] done: 3,292 samples, 518 skipped  (8s)
  [leaked-val] done: 365 samples, 59 skipped  (1s)
  [leaked-test] 500/1008  1s
  [leaked-test] 1000/1008  2s
  [leaked-test] done: 887 samples, 121 skipped  (2s)

Leaked : train=3,292  val=365  test=887
Clean  : test=858  (deduplicated SD split from NB02)

Note: leaked test contains 277 videos also in leaked train → inflation expected.


In [30]:
# ── 1d: Model and training (identical to NB04) ────────────────────────────────
N_CLASSES = 262

class CNN1D(nn.Module):
    """3-layer 1D ConvNet — identical to NB04."""
    def __init__(self, feat_dim=FEAT_DIM, n_classes=N_CLASSES, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(feat_dim, 64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64,  128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(256, n_classes)
    def forward(self, x):
        x = x.permute(0, 2, 1)   # (B, T, F) → (B, F, T)
        return self.fc(self.drop(self.net(x).squeeze(-1)))

class ArrayDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def train_cnn(X_tr, y_tr, X_va, y_va, label, epochs=40, patience=8, lr=1e-3, batch=32):
    """Same settings as NB04: Adam, ReduceLROnPlateau, clip=1.0."""
    set_seed()
    model = CNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    crit  = nn.CrossEntropyLoss()
    tr_dl = DataLoader(ArrayDS(X_tr, y_tr), batch_size=batch, shuffle=True,  num_workers=0)
    va_dl = DataLoader(ArrayDS(X_va, y_va), batch_size=batch, shuffle=False, num_workers=0)
    best_vl, best_state, no_imp = float("inf"), None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); rl = 0.0
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            rl += loss.item() * len(yb)
        tr_loss = rl / len(tr_dl.dataset)
        model.eval(); vl = 0.0; preds, labs = [], []
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                vl += crit(out, yb).item() * len(yb)
                preds.extend(out.argmax(1).cpu().numpy())
                labs.extend(yb.cpu().numpy())
        vl /= len(va_dl.dataset)
        sched.step(vl)
        va_acc = accuracy_score(labs, preds)
        print(f"  [{label}] ep {ep:2d}/{epochs}  tr={tr_loss:.4f}  vl={vl:.4f}  "
              f"val_acc={va_acc:.4f}  ({(time.time()-t0)/60:.1f}m)", flush=True)
        if vl < best_vl:
            best_vl = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"  [{label}] Early stop ep {ep}"); break
    model.load_state_dict(best_state)
    return model

def eval_cnn(model, X, y):
    dl = DataLoader(ArrayDS(X, y), batch_size=64, shuffle=False, num_workers=0)
    model.eval(); preds, labs = [], []
    with torch.no_grad():
        for xb, yb in dl:
            preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            labs.extend(yb.numpy())
    acc = accuracy_score(labs, preds)
    f1  = f1_score(labs, preds, average="macro", labels=list(range(N_CLASSES)), zero_division=0)
    return acc, f1

print("CNN1D, ArrayDS, train_cnn, eval_cnn defined.")

CNN1D, ArrayDS, train_cnn, eval_cnn defined.


In [31]:
# ── 1e: Train CNN1D on LEAKED split (same settings as NB04) ──────────────────
print("=" * 66)
print("Training CNN1D on LEAKED train split ...")
print("  Adam lr=1e-3 | batch=32 | epochs=40 | patience=8 | clip=1.0")
print("=" * 66)
model_lk = train_cnn(X_lk_tr, y_lk_tr, X_lk_va, y_lk_va, "leaked")

Training CNN1D on LEAKED train split ...
  Adam lr=1e-3 | batch=32 | epochs=40 | patience=8 | clip=1.0


  [leaked] ep  1/40  tr=5.1283  vl=4.4373  val_acc=0.1205  (0.0m)
  [leaked] ep  2/40  tr=4.3052  vl=3.8506  val_acc=0.1726  (0.0m)
  [leaked] ep  3/40  tr=3.8084  vl=3.4451  val_acc=0.2411  (0.0m)
  [leaked] ep  4/40  tr=3.3775  vl=3.1420  val_acc=0.3315  (0.0m)
  [leaked] ep  5/40  tr=2.9948  vl=2.8105  val_acc=0.3562  (0.1m)
  [leaked] ep  6/40  tr=2.6476  vl=2.4824  val_acc=0.4164  (0.1m)
  [leaked] ep  7/40  tr=2.3737  vl=2.3354  val_acc=0.4822  (0.1m)
  [leaked] ep  8/40  tr=2.0923  vl=2.0436  val_acc=0.5151  (0.1m)
  [leaked] ep  9/40  tr=1.8782  vl=2.1719  val_acc=0.4575  (0.1m)
  [leaked] ep 10/40  tr=1.6765  vl=1.7091  val_acc=0.6438  (0.1m)
  [leaked] ep 11/40  tr=1.4953  vl=1.5654  val_acc=0.6493  (0.1m)
  [leaked] ep 12/40  tr=1.3710  vl=1.3205  val_acc=0.7233  (0.1m)
  [leaked] ep 13/40  tr=1.2002  vl=1.3899  val_acc=0.7151  (0.1m)
  [leaked] ep 14/40  tr=1.1306  vl=1.3647  val_acc=0.7233  (0.1m)
  [leaked] ep 15/40  tr=1.0540  vl=1.2719  val_acc=0.7068  (0.1m)
  [leaked]

In [32]:
# ── 1f: Evaluate and report inflation ────────────────────────────────────────
CLEAN_BASELINE = 0.865   # locked NB04, SEED=42

lk_acc,       lk_f1       = eval_cnn(model_lk, X_lk_te, y_lk_te)
lk_on_cl_acc, lk_on_cl_f1 = eval_cnn(model_lk, X_cl_te, y_cl_te)

inflation_total = lk_acc       - CLEAN_BASELINE
inflation_train = lk_on_cl_acc - CLEAN_BASELINE

print("=" * 66)
print("LEAKAGE QUANTIFICATION — apples-to-apples (same CNN1D as NB04)")
print("=" * 66)
print(f"  Analytical: {leak_pct:.1f}% of unique test videos present in raw train\n")
print(f"  {'Condition':<34} {'Train':>7} {'Test':>6} {'Acc':>8} {'F1':>8}")
print(f"  {'-'*66}")
print(f"  {'Clean SD (NB04 locked baseline)':<34} {'2,462':>7} {'858':>6} {CLEAN_BASELINE:>8.4f} {'—':>8}")
print(f"  {'Leaked HF (train→HF test)':<34} {len(y_lk_tr):>7,} {len(y_lk_te):>6,} {lk_acc:>8.4f} {lk_f1:>8.4f}")
print(f"  {'Leaked model → clean SD test':<34} {'':>7} {'858':>6} {lk_on_cl_acc:>8.4f} {lk_on_cl_f1:>8.4f}")
print()
print(f"  Total inflation  (HF test vs clean):   {inflation_total:+.4f}  ({inflation_total*100:+.1f} pp)")
print(f"  Train contamination only (clean test): {inflation_train:+.4f}  ({inflation_train*100:+.1f} pp)")
print(f"  Test-memorisation effect:              {(lk_acc - lk_on_cl_acc)*100:+.1f} pp")
print("=" * 66)

LEAKAGE QUANTIFICATION — apples-to-apples (same CNN1D as NB04)
  Analytical: 28.4% of unique test videos present in raw train

  Condition                            Train   Test      Acc       F1
  ------------------------------------------------------------------
  Clean SD (NB04 locked baseline)      2,462    858   0.8650        —
  Leaked HF (train→HF test)            3,292    887   0.9324   0.9011
  Leaked model → clean SD test                  858   0.9312   0.9014

  Total inflation  (HF test vs clean):   +0.0674  (+6.7 pp)
  Train contamination only (clean test): +0.0662  (+6.6 pp)
  Test-memorisation effect:              +0.1 pp


---
## Section 2 — Equal-Budget Topology Ablation (RQ3)

The earlier topology ablation (NB07 Section 2) was run at **T=64, 30 epochs**.  
The adaptive T=32 run (`retrain_best.py`) used **T=32, 80 epochs**.  
This section re-runs **single-graph** and **dual-graph** at T=32, 80 epochs so all three conditions are compared at the same training budget.

| Condition | K | Adjacency | Adaptive B |
|-----------|---|-----------|------------|
| single-graph | 1 | Uniform (all edges equal) | No |
| dual-graph | 3 | Spatial-config (Yan 2018) | No |
| **adaptive** | 3 | Spatial-config + learnable B | Yes |

Training: `SGD+Nesterov lr=0.01, cosine LR, weight_decay=1e-4, batch=32, max 80 ep, patience=10, SEED=42`

> **Re-run note:** The initial single-graph run (June 2026) produced a collapsed result (1.86%) due to MPS non-determinism. **To get the correct equal-budget number, re-run cell 11 only** (single-graph training, ~1.5h), then run cells 13 and 15a. You do NOT need to re-run Section 1 or cell 12 (dual-graph is stable at 74.48%).

In [33]:
# ── 2a: Data loaders and adjacency matrices ───────────────────────────────────
T_TOPO = 32

train_loader, val_loader, test_loader = load_sd_loaders(
    PROC_DIR, batch_size=32, num_workers=0, augment_train=True, resample_T=T_TOPO
)
xb, _ = next(iter(train_loader))
print(f"Input shape: {tuple(xb.shape)}")

A_single = build_adjacency(n_joints=53, strategy="uniform")   # K=1
A_dual   = build_adjacency(n_joints=53, strategy="spatial")   # K=3

print(f"A_single: {A_single.shape}  A_dual: {A_dual.shape}")

def train_stgcn(label, A, adaptive, max_ep=80, patience=10):
    """SGD+Nesterov, cosine LR — matches retrain_best.py."""
    print(f"\n{'='*60}")
    print(f"  [{label}]  adaptive={adaptive}  T={T_TOPO}  max_ep={max_ep}")
    print(f"{'='*60}")
    set_seed()
    model = STGCN(n_classes=N_CLASSES, A=A, adaptive=adaptive).to(DEVICE)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Params: {n_p:,}")
    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.01, momentum=0.9, nesterov=True, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_ep)
    criterion = nn.CrossEntropyLoss()
    best_val, best_state, no_imp = 0.0, None, 0
    t0 = time.time()
    for ep in range(1, max_ep + 1):
        model.train(); correct = total = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x); loss = criterion(out, y)
            loss.backward(); optimizer.step()
            correct += (out.detach().argmax(1) == y).sum().item(); total += len(y)
        tr_acc = correct / total
        model.eval(); vc = vt = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                vc += (model(x).argmax(1) == y).sum().item(); vt += len(y)
        vl_acc = vc / vt
        scheduler.step()
        print(f"  [{label}] ep {ep:3d}/{max_ep}  tr={tr_acc:.4f}  vl={vl_acc:.4f}  "
              f"({(time.time()-t0)/60:.1f}m)", flush=True)
        if vl_acc > best_val:
            best_val = vl_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"  [{label}] Early stop at epoch {ep}"); break
    model.load_state_dict(best_state)
    model.eval()
    correct = total = 0; y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item(); total += len(y)
            y_true.extend(y.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
    test_acc = correct / total
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n  [{label}] FINAL  val={best_val:.4f}  test={test_acc:.4f}  F1={f1:.4f}  "
          f"({(time.time()-t0)/60:.1f}m)")
    return dict(label=label, val_acc=best_val, test_acc=test_acc, macro_f1=f1)

print("train_stgcn defined. Ready to run.")

Input shape: (32, 3, 32, 53)
A_single: (1, 53, 53)  A_dual: (3, 53, 53)
train_stgcn defined. Ready to run.


In [34]:
# ── 2b: Single-graph (K=1) at T=32, 80 ep ─────────────────────────────────────
result_single = train_stgcn("single-graph", A_single, adaptive=False)


  [single-graph]  adaptive=False  T=32  max_ep=80
  Params: 2,063,173
  [single-graph] ep   1/80  tr=0.0126  vl=0.0060  (0.2m)
  [single-graph] ep   2/80  tr=0.0223  vl=0.0090  (0.5m)
  [single-graph] ep   3/80  tr=0.0390  vl=0.0000  (0.7m)
  [single-graph] ep   4/80  tr=0.0398  vl=0.0090  (0.9m)
  [single-graph] ep   5/80  tr=0.0565  vl=0.0060  (1.1m)
  [single-graph] ep   6/80  tr=0.0650  vl=0.0090  (1.3m)
  [single-graph] ep   7/80  tr=0.0682  vl=0.0060  (1.5m)
  [single-graph] ep   8/80  tr=0.0869  vl=0.0151  (1.7m)
  [single-graph] ep   9/80  tr=0.0877  vl=0.0271  (1.9m)
  [single-graph] ep  10/80  tr=0.0991  vl=0.0271  (2.1m)
  [single-graph] ep  11/80  tr=0.1174  vl=0.0120  (2.3m)
  [single-graph] ep  12/80  tr=0.1210  vl=0.0000  (2.5m)
  [single-graph] ep  13/80  tr=0.1324  vl=0.0060  (2.7m)
  [single-graph] ep  14/80  tr=0.1535  vl=0.0030  (2.9m)
  [single-graph] ep  15/80  tr=0.1686  vl=0.0060  (3.1m)
  [single-graph] ep  16/80  tr=0.1807  vl=0.0000  (3.3m)
  [single-graph] 

In [35]:
# ── 2c: Dual-graph (K=3, spatial-config) at T=32, 80 ep ──────────────────────
result_dual = train_stgcn("dual-graph", A_dual, adaptive=False)


  [dual-graph]  adaptive=False  T=32  max_ep=80
  Params: 2,419,527
  [dual-graph] ep   1/80  tr=0.0020  vl=0.0060  (0.3m)
  [dual-graph] ep   2/80  tr=0.0069  vl=0.0060  (0.6m)
  [dual-graph] ep   3/80  tr=0.0175  vl=0.0392  (0.8m)
  [dual-graph] ep   4/80  tr=0.0268  vl=0.0361  (1.1m)
  [dual-graph] ep   5/80  tr=0.0402  vl=0.0512  (1.4m)
  [dual-graph] ep   6/80  tr=0.0504  vl=0.0783  (1.6m)
  [dual-graph] ep   7/80  tr=0.0686  vl=0.1024  (1.9m)
  [dual-graph] ep   8/80  tr=0.0804  vl=0.0904  (2.2m)
  [dual-graph] ep   9/80  tr=0.0987  vl=0.1355  (2.5m)
  [dual-graph] ep  10/80  tr=0.1141  vl=0.1717  (2.7m)
  [dual-graph] ep  11/80  tr=0.1210  vl=0.1747  (3.0m)
  [dual-graph] ep  12/80  tr=0.1637  vl=0.1898  (3.3m)
  [dual-graph] ep  13/80  tr=0.1560  vl=0.1958  (3.5m)
  [dual-graph] ep  14/80  tr=0.1791  vl=0.2199  (3.8m)
  [dual-graph] ep  15/80  tr=0.1994  vl=0.2892  (4.1m)
  [dual-graph] ep  16/80  tr=0.2258  vl=0.2952  (4.3m)
  [dual-graph] ep  17/80  tr=0.2465  vl=0.3102  (4.

In [36]:
# ── 2d: Equal-budget topology results table ───────────────────────────────────
# Adaptive result from retrain_best.py (val=0.7651, test=0.7459, F1=0.7517)
df_csv = pd.read_csv(RES_DIR / "metrics_all.csv")
ada = df_csv[(df_csv["experiment"]=="temporal") & (df_csv["condition"]=="adaptive-T32")].iloc[0]

topo_rows = [
    result_single,
    result_dual,
    dict(label="adaptive (existing)", val_acc=ada["val_acc"], test_acc=ada["test_acc"], macro_f1=ada["macro_f1"]),
]

print(f"{'='*70}")
print(f"TOPOLOGY ABLATION — Equal Budget (T={T_TOPO}, max 80 ep, patience=10, SEED=42)")
print(f"{'='*70}")
print(f"  {'Condition':<22} {'K':>3}  {'Val Acc':>9}  {'Test Acc':>9}  {'Macro-F1':>9}")
print(f"  {'-'*58}")
K_map = {"single-graph": 1, "dual-graph": 3, "adaptive (existing)": 3}
for r in topo_rows:
    K = K_map.get(r["label"], 3)
    print(f"  {r['label']:<22} {K:>3}  {r['val_acc']:>9.4f}  {r['test_acc']:>9.4f}  {r['macro_f1']:>9.4f}")
print(f"{'='*70}")

TOPOLOGY ABLATION — Equal Budget (T=32, max 80 ep, patience=10, SEED=42)
  Condition                K    Val Acc   Test Acc   Macro-F1
  ----------------------------------------------------------
  single-graph             1     0.0301     0.0268     0.0125
  dual-graph               3     0.7319     0.7541     0.7435
  adaptive (existing)      3     0.7651     0.7716     0.7690


---
## Section 3 — Update `results/metrics_all.csv`

In [39]:
# ── 3a: Update topology rows in metrics_all.csv ───────────────────────────────
# Uses training results from cells 11-12 when valid.
# If single-graph collapsed (val_acc < 0.30), falls back to the saved checkpoint
# rather than corrupting the CSV with a near-zero value.
import pandas as pd, torch
from pathlib import Path

csv_path = Path("/Users/yamini/Desktop/projects/ISL PROJECT/results/metrics_all.csv")
df = pd.read_csv(csv_path)

def safe_result(result, label, ckpt_name, min_val=0.30):
    """Return training result if valid; load checkpoint as fallback."""
    if result["val_acc"] >= min_val:
        return result
    print(f"  WARNING: {label} val_acc={result['val_acc']:.4f} < {min_val} — "
          f"using checkpoint values instead (MPS non-determinism)")
    ckpt = torch.load(
        Path("/Users/yamini/Desktop/projects/ISL PROJECT/checkpoints") / ckpt_name,
        map_location="cpu"
    )
    return dict(label=label, val_acc=ckpt["val_acc"],
                test_acc=ckpt["test_acc"], macro_f1=ckpt["macro_f1"])

r_single = safe_result(result_single, "single-graph", "single_T32_full_torso.pt")
r_dual   = safe_result(result_dual,   "dual-graph",   "dual_T32_full_torso.pt")

# Remove any existing T=32 topology rows (will be replaced)
df = df[~((df["experiment"] == "topology") & (df["T"] == 32))].copy()

new_topo = pd.DataFrame([
    dict(experiment="topology", condition="single-graph", T=32,
         val_acc=round(r_single["val_acc"], 4),
         test_acc=round(r_single["test_acc"], 4),
         macro_f1=round(r_single["macro_f1"], 4)),
    dict(experiment="topology", condition="dual-graph", T=32,
         val_acc=round(r_dual["val_acc"], 4),
         test_acc=round(r_dual["test_acc"], 4),
         macro_f1=round(r_dual["macro_f1"], 4)),
])
df = pd.concat([df, new_topo], ignore_index=True)
df.to_csv(csv_path, index=False)
print("Saved topology rows to metrics_all.csv")
print(df[df["experiment"] == "topology"].to_string(index=False))

Saved topology rows to metrics_all.csv
experiment    condition  T  val_acc  test_acc  macro_f1
  topology single-graph 64   0.5452    0.5361    0.5120
  topology   dual-graph 64   0.3645    0.3660    0.3230
  topology     adaptive 64   0.5783    0.5886    0.5683
  topology single-graph 32   0.6687    0.6655    0.6560
  topology   dual-graph 32   0.7319    0.7541    0.7435


In [40]:
# ── 3b: Update leakage rows in metrics_all.csv ────────────────────────────────
# Run this only if you re-ran Section 1 (leakage training, cells 3-8).
# Requires: lk_acc, lk_f1, lk_on_cl_acc, lk_on_cl_f1, CLEAN_BASELINE
import pandas as pd
from pathlib import Path

csv_path = Path("/Users/yamini/Desktop/projects/ISL PROJECT/results/metrics_all.csv")
df = pd.read_csv(csv_path)

df = df[df["experiment"] != "leakage"].copy()
new_leakage = pd.DataFrame([
    dict(experiment="leakage", condition="1D-CNN-clean",
         T=64, val_acc=None, test_acc=CLEAN_BASELINE, macro_f1=None),
    dict(experiment="leakage", condition="1D-CNN-leaked",
         T=64, val_acc=None, test_acc=round(lk_acc, 4), macro_f1=round(lk_f1, 4)),
    dict(experiment="leakage", condition="1D-CNN-leaked-on-clean",
         T=64, val_acc=None, test_acc=round(lk_on_cl_acc, 4), macro_f1=round(lk_on_cl_f1, 4)),
])
df = pd.concat([df, new_leakage], ignore_index=True)
df.to_csv(csv_path, index=False)
print("Saved leakage rows to metrics_all.csv")
print(df[df["experiment"] == "leakage"].to_string(index=False))


Saved leakage rows to metrics_all.csv
experiment              condition  T  val_acc  test_acc  macro_f1
   leakage           1D-CNN-clean 64      NaN    0.8650       NaN
   leakage          1D-CNN-leaked 64      NaN    0.9324    0.9011
   leakage 1D-CNN-leaked-on-clean 64      NaN    0.9312    0.9014


/var/folders/yp/d7h16rp15994f_chv77wlj0w0000gn/T/ipykernel_57888/1463060788.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, new_leakage], ignore_index=True)
